# Milan, Sentinel-2 (percentile), GHSL — all points, no train/test split

Test variant requested alongside the AlphaEarth and GHSL tests: every point in
`./output/milan/ghsl/percentile`'s train+test split is combined and used for both spatial-block CV
tuning and the final fit. RF only. Reuses the existing extracted S2 bands
directly (`spatial_train_pts.gpkg` + `spatial_test_pts.gpkg` concatenated) --
this is exactly "all points", since train+test together is everything notebook
00/01b ever sampled, and avoids re-extracting from GEE.

No holdout exists here by design; evaluate against the independent EarthLabel
ground truth instead (see the project's validation pipeline), not any
in-sample figure printed below.

## Cell 1 · Imports & configuration

In [1]:
import ee
from gee_init import init_gee

init_gee()

import json
import os
import time
import warnings

import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

from s2_utils import build_composite, water_mask

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS       = -1
N_FOLDS      = 5
BLOCK_SIZES  = [500, 1000, 2000]
N_ITER_RF    = 40

TARGET    = 'GHSL'
METHOD    = 'percentile'
LABEL_COL = 'GHSL'
SRC_DIR   = './output/milan/ghsl/percentile'
OUTPUT_DIR = './output/milan/ghsl/percentile_alltrain'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DRIVE_FOLDER = 'GEE_exports'

print(f'Milan, S2 {METHOD}, {TARGET} -- ALL POINTS, no train/test split')
print(f'source: {SRC_DIR}  ->  output: {OUTPUT_DIR}')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


GEE initialised (project: imd-vietnam).


Milan, S2 percentile, GHSL -- ALL POINTS, no train/test split
source: ./output/milan/ghsl/percentile  ->  output: ./output/milan/ghsl/percentile_alltrain


---
## Cell 2 · AOI, S2 composite, all sample points (reused from the existing split)

In [2]:
from shapely.geometry import mapping

aoi_gdf  = gpd.read_file('./data/aoi_milan/milano_aoi.gpkg').to_crs('EPSG:4326')
aoi_geom = ee.Geometry(mapping(aoi_gdf.geometry.union_all()))
print(f'Milan AOI area: {aoi_geom.area(1).getInfo() / 1e6:,.0f} km2')

non_water_mask = water_mask(aoi_geom)

# model_metadata_S2.json, not s2_extraction_metadata.json -- the latter only
# exists under clms/, not ghsl/, while this one exists (with identical dates)
# under both.
with open(f'{SRC_DIR}/model_metadata_S2.json') as f:
    s2_meta = json.load(f)
SELECTED_DATES = s2_meta['selected_dates']
PERCENTILES = (10, 25, 50, 75, 90)   # ignored by 'median'/'stack'

pred_source_img, s2_proj, band_names = build_composite(
    'COPERNICUS/S2_SR_HARMONIZED', SELECTED_DATES, aoi_geom, non_water_mask,
    method=METHOD, percentiles=PERCENTILES)
print(f'S2 bands: {len(band_names)}  dates: {len(SELECTED_DATES)}')

train_gdf = gpd.read_file(f'{SRC_DIR}/spatial_train_pts.gpkg')
test_gdf  = gpd.read_file(f'{SRC_DIR}/spatial_test_pts.gpkg')
gdf_all = pd.concat([train_gdf, test_gdf], ignore_index=True)
gdf_all = gpd.GeoDataFrame(gdf_all, geometry='geometry', crs=train_gdf.crs)
assert set(band_names) == set(c for c in gdf_all.columns if c in band_names), (
    'Band set mismatch between the rebuilt composite and the existing table.')

gdf_utm = gdf_all.to_crs('EPSG:32632')
coords_all = np.column_stack([gdf_utm.geometry.x, gdf_utm.geometry.y])
X_all = gdf_all[band_names].values.astype(np.float32)
y_all = gdf_all[LABEL_COL].values.astype(np.float32)

print(f'Total points (ALL used for training): {len(gdf_all)} '
      f'(train {len(train_gdf)} + test {len(test_gdf)})')
print(f'X shape: {X_all.shape} | y range: {y_all.min():.0f}-{y_all.max():.0f}%')

Milan AOI area: 4,455 km2
S2 bands: 50  dates: 30
Total points (ALL used for training): 3448 (train 2450 + test 998)
X shape: (3448, 50) | y range: 0-100%


---
## Cell 3 · Spatial block helper functions

In [3]:
def assign_blocks(coords, block_size):
    bx = (coords[:, 0] // block_size).astype(int)
    by = (coords[:, 1] // block_size).astype(int)
    return np.array([f'{x}_{y}' for x, y in zip(bx, by)])

def blocks_to_folds(block_ids, n_folds, seed):
    rng    = np.random.RandomState(seed)
    unique = rng.permutation(np.unique(block_ids))
    bmap   = {b: i % n_folds for i, b in enumerate(unique)}
    return np.array([bmap[b] for b in block_ids])

def make_spatial_splits(point_folds, n_folds):
    all_idx = np.arange(len(point_folds))
    return [(all_idx[point_folds != f], all_idx[point_folds == f])
            for f in range(n_folds)]

print('Spatial block functions defined.')

Spatial block functions defined.


---
## Cell 4 · RF hyperparameter tuning, 5-fold spatial-block CV, on ALL points

In [4]:
PARAM_SPACE = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'max_samples':       [None, 0.7, 0.8, 0.9],
}

best_rmse_tune, best_search, best_block = np.inf, None, None
for bs in BLOCK_SIZES:
    bids = assign_blocks(coords_all, bs)
    pf = blocks_to_folds(bids, N_FOLDS, RANDOM_STATE)
    cv_splits = make_spatial_splits(pf, N_FOLDS)
    t0 = time.time()
    search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        param_distributions=PARAM_SPACE, n_iter=N_ITER_RF, cv=cv_splits,
        scoring='neg_root_mean_squared_error', n_jobs=N_JOBS,
        random_state=RANDOM_STATE, refit=True)
    search.fit(X_all, y_all)
    cv_rmse = -search.best_score_
    print(f'{bs}m spatial CV -> RMSE={cv_rmse:.3f}  ({time.time() - t0:.1f}s)')
    if cv_rmse < best_rmse_tune:
        best_rmse_tune, best_search, best_block = cv_rmse, search, bs

best_params = best_search.best_params_
print(f'\nBEST: block={best_block}m  CV_RMSE={best_rmse_tune:.3f}')
print(f'params={best_params}')

500m spatial CV -> RMSE=18.574  (346.3s)


1000m spatial CV -> RMSE=18.742  (341.5s)


2000m spatial CV -> RMSE=18.716  (340.9s)

BEST: block=500m  CV_RMSE=18.574
params={'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_samples': 0.7, 'max_features': 0.3, 'max_depth': 10}


---
## Cell 5 · Fit final RF on all points, export the predicted raster

In [5]:
rf_fitted = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, **best_params)
rf_fitted.fit(X_all, y_all)
joblib.dump(rf_fitted, f'{OUTPUT_DIR}/best_model_RF_alltrain.joblib')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_fit = np.clip(rf_fitted.predict(X_all), 0, 100)
print('In-sample fit (NOT a holdout metric -- these points were also used to train):')
print(f'  RMSE={np.sqrt(mean_squared_error(y_all, y_fit)):.2f}  '
      f'MAE={mean_absolute_error(y_all, y_fit):.2f}  R2={r2_score(y_all, y_fit):.3f}')

features = [ee.Feature(None, {**{b: float(row[b]) for b in band_names}, 'IMD': float(row[LABEL_COL])})
            for _, row in gdf_all.iterrows()]
train_fc = ee.FeatureCollection(features)

clf = (ee.Classifier.smileRandomForest(
           numberOfTrees=best_params.get('n_estimators', 200),
           minLeafPopulation=best_params.get('min_samples_leaf', 1),
           seed=RANDOM_STATE)
       .setOutputMode('REGRESSION')
       .train(features=train_fc, classProperty='IMD', inputProperties=band_names))

pred_img = (pred_source_img.classify(clf).rename('IMD_predicted')
            .clip(aoi_geom).updateMask(non_water_mask).clamp(0, 100))

fname = 'GHSL_predicted_RF_S2_percentile_alltrain_Milan'
out_tif = f'{OUTPUT_DIR}/{fname}.tif'
if os.path.exists(out_tif):
    print(f'raster already present at {out_tif} -- skipping export.')
else:
    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_geom, scale=10, crs='EPSG:32632',
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'  {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'export {state} -- {task.status()}')

    creds = ee.data.get_persistent_credentials()
    drive = gbuild('drive', 'v3', credentials=creds)
    resp = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                               fields='files(id, createdTime)').execute()
    files = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'downloaded -> {out_tif}')

meta = {
    'band_names': band_names, 'n_points_train': len(gdf_all),
    'best_block_m': best_block, 'cv_rmse_tune': best_rmse_tune, 'best_params': best_params,
    'in_sample_rmse': float(np.sqrt(mean_squared_error(y_all, y_fit))),
    'in_sample_mae': float(mean_absolute_error(y_all, y_fit)),
    'in_sample_r2': float(r2_score(y_all, y_fit)),
    'note': 'No train/test split -- all points used for both tuning and the final fit. '
            'in_sample_* are NOT holdout metrics. Evaluate against EarthLabel instead.',
}
with open(f'{OUTPUT_DIR}/model_metadata_alltrain.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f'\nSaved -> {OUTPUT_DIR}/model_metadata_alltrain.json')

In-sample fit (NOT a holdout metric -- these points were also used to train):
  RMSE=13.41  MAE=9.66  R2=0.863


submitted batch export task (id=KNHYGDWK7423CYOJTKUYWOSN)


  RUNNING


  RUNNING


  RUNNING


...

  RUNNING


  RUNNING


  RUNNING


  COMPLETED


downloaded -> ./output/milan/ghsl/percentile_alltrain/GHSL_predicted_RF_S2_percentile_alltrain_Milan.tif

Saved -> ./output/milan/ghsl/percentile_alltrain/model_metadata_alltrain.json


---
## Cell 6 · Feature importance (permutation, in-sample)

No holdout exists for this run by design, so this ranks which bands the
final model relies on most -- not a generalisation measure, but informative
for comparing band importance across runs. Computed on the same `X_all`
(or `X_train`, for a split-mode notebook) the model was fit on.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.inspection import permutation_importance

result = permutation_importance(rf_fitted, X_all, y_all, n_repeats=10,
                                 random_state=RANDOM_STATE,
                                 scoring='neg_root_mean_squared_error', n_jobs=N_JOBS)
imp_df = pd.DataFrame({
    'band': band_names,
    'importance_mean': result.importances_mean,
    'importance_std': result.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)
imp_df.to_csv(f'{OUTPUT_DIR}/feature_importance_RF_alltrain.csv', index=False)

top = imp_df.head(20)
fig, ax = plt.subplots(figsize=(7, max(4, 0.3 * len(top))))
ax.barh(top['band'][::-1], top['importance_mean'][::-1], xerr=top['importance_std'][::-1], color='#2ecc71')
ax.set_xlabel('Permutation importance (RMSE increase, in-sample)')
ax.set_title('Feature importance, RF, all points')
fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/figD_importance_RF_alltrain.png', dpi=140)
plt.show()

print(imp_df.head(10).to_string(index=False))